# Cut text-to-SQL prompt tokens with Mistral by sending only the tables a question needs

A text-to-SQL prompt carries the database schema. Send all of it and every question pays for every
table, on every call. On a real warehouse that is tens of thousands of tokens, most of them about
tables the question never touches.

This notebook sends the model only the tables a question needs, picked by
[schemagate](https://github.com/ashishsinha1602/schemagate), an open-source selector that runs
offline (BM25 plus a small embedder, no model call). A small model is often good enough for SQL once the prompt holds only the tables that matter.

What you will see:

1. the full schema and the selected schema sent to the same model, with the input tokens the API
   reports for each;
2. the SQL from the selection run against a real SQLite database, next to the SQL from the full
   schema;
3. the selection respecting who is asking: a table the caller may not read never enters the prompt;
4. what the saving is worth per month, and when it is not worth much.

You need a Mistral API key from [console.mistral.ai](https://console.mistral.ai/api-keys); the free tier is enough. The notebook makes two
model calls.

In [ ]:
%pip install -q "schemagate>=1.15" "mistralai>=3"

In [ ]:
import os
import sqlite3

from schemagate import Principal
from schemagate.demo_schema import demo_catalog

from mistralai.client import Mistral

client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

MODEL = os.environ.get("MISTRAL_MODEL", "mistral-small-latest")

## 1. A schema with more tables than any question needs

The bundled demo schema has 42 tables and views across a commerce, billing, HR and inventory
database, with the traps real schemas have: near-duplicate tables, cryptic legacy names, join tables
a question never mentions. It is materialised as a SQLite file with a few rows, so the SQL can run.

In [ ]:
cat = demo_catalog()
objects = list(cat.objects())
full_schema = "\n\n".join(o.render_ddl(max_columns=10**6) for o in objects)
db = sqlite3.connect(cat._demo_url.removeprefix("sqlite:///"))
print(len(objects), "objects,", len(full_schema), "characters of DDL")

## 2. Select the tables for one question

`select` ranks every table and view against the question and keeps the top six plus the tables they
join to. The `principal` is who is asking; section 4 shows why that matters.

In [ ]:
QUESTION = "Which customers owe us money, and how much?"

finance = Principal("okta:analyst", roles={"finance"})
selection = cat.select(QUESTION, top_k=6, principal=finance)
selected_schema = selection.prompt_fragment()
print("tables sent:", [t.split(".")[-1] for t in selection.table_names])

## 3. Ask Mistral twice: full schema, then the selection

The prompt is identical except for the schema block. The token counts are the ones the API returns
in `usage.prompt_tokens`, so they come from the model's own tokenizer. Anything the API adds to every
request is in both counts, so the difference between them is the schema alone. Both queries run against the
demo database, so you can check each answer is real, not just plausible.

In [ ]:
def prompt(schema: str, question: str) -> str:
    return (
        f"Database schema (SQLite):\n\n{schema}\n\nQuestion: {question}\n\n"
        "Reply with one SQLite SELECT and nothing else."
    )


def ask(schema: str, question: str):
    response = client.chat.complete(
        model=MODEL, messages=[{"role": "user", "content": prompt(schema, question)}]
    )
    text = response.choices[0].message.content
    sql = text.strip().removeprefix("```sql").removeprefix("```").removesuffix("```").strip()
    return sql, response.usage.prompt_tokens


def run(sql: str):
    try:
        return db.execute(sql).fetchall()
    except sqlite3.Error as e:
        return f"error: {e}"


full_sql, full_tokens = ask(full_schema, QUESTION)
small_sql, small_tokens = ask(selected_schema, QUESTION)

print(f"full schema: {full_tokens:,} input tokens")
print(f"selection:   {small_tokens:,} input tokens  ({1 - small_tokens / full_tokens:.0%} fewer)")

In [ ]:
for label, sql in (("selection", small_sql), ("full schema", full_sql)):
    rows = run(sql)
    print(f"--- {label}\n{sql}")
    if isinstance(rows, str):
        print(rows)
    else:
        print(len(rows), "rows, first five:")
        for row in rows[:5]:
            print(tuple(round(v, 2) if isinstance(v, float) else v for v in row))

Model output varies from run to run, so the two queries may differ in column names or rounding.
What to check is that the selection's query reads the right tables and returns the debts.

## 4. Selection is also access control

`hr_compensation` is readable only by the `payroll` role. For anyone else it isn't just ranked low:
it is left out entirely, so the model can't write a query against it or mention that it exists.

In [ ]:
SALARY = "salary and pay grade by employee"
for who in (Principal("okta:analyst", roles={"finance"}), Principal("okta:hr", roles={"payroll"})):
    tables = [t.split(".")[-1] for t in cat.select(SALARY, top_k=6, principal=who).table_names]
    print(sorted(who.roles), "->", "hr_compensation" in tables, tables)

## 5. What it is worth, and when it is not

Replace the volume and the price with your own. The selection stays at a handful of tables (the top
six, plus the tables they join to) however large the schema grows, so the saving grows with the
schema.

In [ ]:
questions_per_month = 1_000_000
price_per_million_input = 0.15  # USD, put your model's input price here


def monthly(tokens_per_question: int) -> float:
    return tokens_per_question * questions_per_month / 1e6 * price_per_million_input


print(f"full schema: ${monthly(full_tokens):,.0f}/month")
print(f"selection:   ${monthly(small_tokens):,.0f}/month")

On public benchmarks, measured the same way but offline (no model called; the script and
per-database numbers are in schemagate's
[BENCHMARKS.md](https://github.com/ashishsinha1602/schemagate/blob/main/BENCHMARKS.md#prompt-tokens-on-public-schemas-the-saving-grows-with-the-schema)):

| Schemas | Median tables | Median saving per database |
|---|---|---|
| BIRD dev | 7 | 8.8% |
| Spider 2.0-lite BigQuery and Snowflake | 11 | 44.3% |
| the same, databases with 50+ tables | | 91.1% |

A seven-table database is already a few hundred tokens, so there is little to cut. Selection pays off
on big schemas. It also has a price: a table the question needs can be left out (on the Spider 2.0
warehouses, every gold table was selected for 86.7% of questions), so measure recall on your own
questions before relying on it.

## Where to go next

- `cat = Catalog().bootstrap("postgresql://...")` reflects your own database instead of the demo.
- `pip install "schemagate[mcp]"` runs the same selection as an MCP server for agents.
- `SchemagateRetriever` (`pip install langchain-schemagate`) does it inside a LangChain or LangGraph
  agent.